# Reconocimiento de Dígitos Manuscritos con Redes Neuronales

## Objetivo

Crear una red neuronal con TensorFlow que reconozca dígitos manuscritos (0-9) del dataset MNIST y permita clasificar imágenes personalizadas dibujadas por el usuario.

**Dataset MNIST:**
- 60,000 imágenes de entrenamiento
- 10,000 imágenes de prueba
- Imágenes en escala de grises de 28x28 píxeles
- 10 clases (dígitos del 0 al 9)

## 1. Instalación de Dependencias

Instalamos TensorFlow y otras librerías necesarias para visualización y procesamiento de imágenes.

In [ ]:
# Instalamos las dependencias necesarias
# Solo es necesario ejecutar esto una vez
%pip install tensorflow matplotlib pillow numpy

## 2. Importación de Librerías

In [ ]:
# Imports necesarios
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from PIL import Image
import os

# Configuración para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow versión: {tf.__version__}")
print(f"Dispositivo disponible: GPU" if len(tf.config.list_physical_devices('GPU')) > 0 else "Dispositivo disponible: CPU")

## 3. Carga y Exploración del Dataset MNIST

MNIST es uno de los datasets más famosos en Machine Learning. Contiene imágenes de dígitos escritos a mano que han sido normalizadas y centradas.

In [ ]:
# Cargamos el dataset MNIST directamente desde Keras
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(f"📊 Datos de entrenamiento: {X_train.shape}")
print(f"📊 Etiquetas de entrenamiento: {y_train.shape}")
print(f"📊 Datos de prueba: {X_test.shape}")
print(f"📊 Etiquetas de prueba: {y_test.shape}")
print(f"\n📈 Rango de valores de píxeles: {X_train.min()} - {X_train.max()}")
print(f"📈 Clases únicas: {np.unique(y_train)}")

### Visualización de Ejemplos del Dataset

In [ ]:
# Visualizamos algunos ejemplos aleatorios
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Ejemplos del Dataset MNIST', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    idx = np.random.randint(0, len(X_train))
    ax.imshow(X_train[idx], cmap='gray')
    ax.set_title(f'Etiqueta: {y_train[idx]}', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

### Distribución de Clases

In [ ]:
# Verificamos que las clases están balanceadas
unique, counts = np.unique(y_train, return_counts=True)

plt.figure(figsize=(10, 5))
plt.bar(unique, counts, color='steelblue', edgecolor='black')
plt.xlabel('Dígito', fontsize=12)
plt.ylabel('Cantidad de muestras', fontsize=12)
plt.title('Distribución de Clases en el Dataset de Entrenamiento', fontsize=14, fontweight='bold')
plt.xticks(unique)
plt.grid(axis='y', alpha=0.3)

for i, count in enumerate(counts):
    plt.text(i, count + 100, str(count), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 4. Preprocesamiento de Datos

Preparamos los datos para entrenar la red neuronal:
1. **Normalización**: Escalamos los píxeles de [0, 255] a [0, 1]
2. **Reshape**: Aplanamos las imágenes de 28x28 a vectores de 784 elementos
3. **One-Hot Encoding**: Convertimos las etiquetas a formato categórico (para la capa de salida softmax)

In [ ]:
# ------------------------------------------------------------------
# 4.1 NORMALIZACIÓN
# ------------------------------------------------------------------
# ¿Por qué normalizar?
#   - Los valores de píxeles están en [0, 255]
#   - Las redes neuronales convergen mejor con valores pequeños
#   - Dividimos por 255.0 para obtener valores en [0, 1]
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

# ------------------------------------------------------------------
# 4.2 RESHAPE (Aplanamiento)
# ------------------------------------------------------------------
# Convertimos las imágenes de 28x28 a vectores de 784 elementos
# Esto es necesario para una red neuronal densa (fully connected)
X_train_flat = X_train_norm.reshape(-1, 28 * 28)  # (60000, 784)
X_test_flat = X_test_norm.reshape(-1, 28 * 28)    # (10000, 784)

# ------------------------------------------------------------------
# 4.3 ONE-HOT ENCODING DE ETIQUETAS
# ------------------------------------------------------------------
# Convertimos las etiquetas de enteros a vectores binarios
# Ejemplo: 3 -> [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

print(f"✅ Forma de X_train_flat: {X_train_flat.shape}")
print(f"✅ Forma de X_test_flat: {X_test_flat.shape}")
print(f"✅ Forma de y_train_cat: {y_train_cat.shape}")
print(f"✅ Forma de y_test_cat: {y_test_cat.shape}")
print(f"\n📌 Ejemplo de etiqueta original: {y_train[0]}")
print(f"📌 Ejemplo de etiqueta one-hot: {y_train_cat[0]}")

## 5. Construcción del Modelo de Red Neuronal

### Arquitectura de la Red:

```
Entrada (784 neuronas) → Capa Oculta 1 (128 neuronas, ReLU)
                       → Dropout (20%)
                       → Capa Oculta 2 (64 neuronas, ReLU)
                       → Dropout (20%)
                       → Salida (10 neuronas, Softmax)
```

**Decisiones de diseño:**
- **Capa de entrada**: 784 neuronas (28×28 píxeles)
- **Capas ocultas**: 128 y 64 neuronas (suficientes para aprender patrones complejos)
- **ReLU**: Función de activación rápida y efectiva
- **Dropout**: Previene sobreajuste desactivando aleatoriamente el 20% de neuronas
- **Softmax**: Convierte las salidas en probabilidades que suman 1

In [ ]:
# Creamos el modelo secuencial
model = models.Sequential([
    # Capa de entrada explícita
    layers.Input(shape=(784,)),
    
    # Primera capa oculta
    layers.Dense(128, activation='relu', name='capa_oculta_1'),
    layers.Dropout(0.2, name='dropout_1'),  # Previene overfitting
    
    # Segunda capa oculta
    layers.Dense(64, activation='relu', name='capa_oculta_2'),
    layers.Dropout(0.2, name='dropout_2'),
    
    # Capa de salida
    layers.Dense(10, activation='softmax', name='salida')
])

# Mostramos la arquitectura del modelo
model.summary()

## 6. Compilación del Modelo

**Configuración:**
- **Optimizador**: Adam (adaptativo, converge rápidamente)
- **Función de pérdida**: Categorical Crossentropy (para clasificación multiclase con one-hot)
- **Métricas**: Accuracy (precisión de clasificación)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Modelo compilado correctamente")

## 7. Entrenamiento del Modelo

**Hiperparámetros:**
- **Epochs**: 10 (iteraciones completas sobre el dataset)
- **Batch size**: 128 (muestras procesadas antes de actualizar pesos)
- **Validation split**: 10% (para monitorear el rendimiento durante el entrenamiento)

In [ ]:
print("🚀 Iniciando entrenamiento...\n")

history = model.fit(
    X_train_flat, 
    y_train_cat,
    epochs=10,
    batch_size=128,
    validation_split=0.1,  # Usamos 10% de los datos de entrenamiento para validación
    verbose=1
)

print("\n✅ Entrenamiento completado")

### Visualización del Entrenamiento

In [ ]:
# Graficamos la evolución de la precisión y pérdida
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de precisión
ax1.plot(history.history['accuracy'], label='Entrenamiento', marker='o', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validación', marker='s', linewidth=2)
ax1.set_title('Evolución de la Precisión', fontsize=14, fontweight='bold')
ax1.set_xlabel('Época', fontsize=12)
ax1.set_ylabel('Precisión', fontsize=12)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Gráfico de pérdida
ax2.plot(history.history['loss'], label='Entrenamiento', marker='o', linewidth=2)
ax2.plot(history.history['val_loss'], label='Validación', marker='s', linewidth=2)
ax2.set_title('Evolución de la Pérdida', fontsize=14, fontweight='bold')
ax2.set_xlabel('Época', fontsize=12)
ax2.set_ylabel('Pérdida', fontsize=12)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Evaluación en el Conjunto de Prueba

In [ ]:
# Evaluamos el modelo en los datos de prueba (nunca vistos durante el entrenamiento)
test_loss, test_accuracy = model.evaluate(X_test_flat, y_test_cat, verbose=0)

print(f"\n📊 RESULTADOS EN EL CONJUNTO DE PRUEBA:")
print(f"   Pérdida: {test_loss:.4f}")
print(f"   Precisión: {test_accuracy * 100:.2f}%")

### Matriz de Confusión

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Obtenemos las predicciones
y_pred = model.predict(X_test_flat)
y_pred_classes = np.argmax(y_pred, axis=1)

# Calculamos la matriz de confusión
cm = confusion_matrix(y_test, y_pred_classes)

# Visualizamos la matriz de confusión
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', square=True, cbar_kws={'label': 'Cantidad'})
plt.title('Matriz de Confusión - Reconocimiento de Dígitos', fontsize=14, fontweight='bold')
plt.ylabel('Etiqueta Real', fontsize=12)
plt.xlabel('Predicción', fontsize=12)
plt.tight_layout()
plt.show()

# Reporte de clasificación
print("\n📈 REPORTE DE CLASIFICACIÓN:")
print(classification_report(y_test, y_pred_classes, digits=4))

### Visualización de Predicciones Correctas e Incorrectas

In [ ]:
# Ejemplos de predicciones correctas
correct_indices = np.where(y_pred_classes == y_test)[0]
incorrect_indices = np.where(y_pred_classes != y_test)[0]

print(f"✅ Predicciones correctas: {len(correct_indices)} de {len(y_test)}")
print(f"❌ Predicciones incorrectas: {len(incorrect_indices)} de {len(y_test)}")

# Mostramos ejemplos aleatorios de predicciones correctas
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('✅ Ejemplos de Predicciones CORRECTAS', fontsize=14, fontweight='bold', color='green')

for i, ax in enumerate(axes.flat):
    idx = np.random.choice(correct_indices)
    ax.imshow(X_test[idx], cmap='gray')
    confidence = y_pred[idx][y_pred_classes[idx]] * 100
    ax.set_title(f'Real: {y_test[idx]} | Pred: {y_pred_classes[idx]}\nConfianza: {confidence:.1f}%', 
                fontsize=10, color='green')
    ax.axis('off')

plt.tight_layout()
plt.show()

# Mostramos ejemplos de predicciones incorrectas (si existen)
if len(incorrect_indices) > 0:
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    fig.suptitle('❌ Ejemplos de Predicciones INCORRECTAS', fontsize=14, fontweight='bold', color='red')
    
    for i, ax in enumerate(axes.flat[:min(10, len(incorrect_indices))]):
        idx = incorrect_indices[i]
        ax.imshow(X_test[idx], cmap='gray')
        confidence = y_pred[idx][y_pred_classes[idx]] * 100
        ax.set_title(f'Real: {y_test[idx]} | Pred: {y_pred_classes[idx]}\nConfianza: {confidence:.1f}%', 
                    fontsize=10, color='red')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

## 9. Guardar el Modelo Entrenado

In [ ]:
# Guardamos el modelo en formato Keras
model.save('modelo_mnist_digitos.keras')
print("💾 Modelo guardado como 'modelo_mnist_digitos.keras'")

# También podemos guardar en formato h5 (formato antiguo pero compatible)
model.save('modelo_mnist_digitos.h5')
print("💾 Modelo guardado como 'modelo_mnist_digitos.h5'")

## 10. Función para Clasificar Imágenes Personalizadas

Esta función permite clasificar imágenes que tú dibujes o cargues desde tu ordenador.

In [ ]:
def clasificar_imagen_personalizada(ruta_imagen, mostrar=True):
    """
    Clasifica una imagen de un dígito manuscrito.
    
    Args:
        ruta_imagen (str): Ruta al archivo de imagen
        mostrar (bool): Si True, muestra la imagen y la predicción
    
    Returns:
        tuple: (dígito_predicho, probabilidades)
    """
    # Cargamos la imagen
    img = Image.open(ruta_imagen).convert('L')  # Convertimos a escala de grises
    
    # Redimensionamos a 28x28 píxeles (tamaño de MNIST)
    img_resized = img.resize((28, 28), Image.Resampling.LANCZOS)
    
    # Convertimos a array numpy
    img_array = np.array(img_resized)
    
    # IMPORTANTE: MNIST tiene fondo negro y dígitos blancos
    # Si tu imagen tiene fondo blanco y dígito negro, invertimos
    if img_array.mean() > 127:  # Fondo más claro que oscuro
        img_array = 255 - img_array  # Invertimos colores
    
    # Normalizamos igual que en el entrenamiento
    img_normalized = img_array.astype('float32') / 255.0
    
    # Aplanamos a vector de 784 elementos
    img_flat = img_normalized.reshape(1, 784)
    
    # Hacemos la predicción
    prediccion = model.predict(img_flat, verbose=0)
    digito_predicho = np.argmax(prediccion)
    confianza = prediccion[0][digito_predicho] * 100
    
    if mostrar:
        # Mostramos la imagen procesada y la predicción
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        
        # Imagen procesada
        ax1.imshow(img_array, cmap='gray')
        ax1.set_title('Imagen Procesada (28x28)', fontsize=12, fontweight='bold')
        ax1.axis('off')
        
        # Gráfico de probabilidades
        ax2.bar(range(10), prediccion[0], color='steelblue', edgecolor='black')
        ax2.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='50% confianza')
        ax2.set_xlabel('Dígito', fontsize=11)
        ax2.set_ylabel('Probabilidad', fontsize=11)
        ax2.set_title(f'Predicción: {digito_predicho} (Confianza: {confianza:.1f}%)', 
                     fontsize=12, fontweight='bold')
        ax2.set_xticks(range(10))
        ax2.grid(axis='y', alpha=0.3)
        ax2.legend()
        
        plt.tight_layout()
        plt.show()
    
    return digito_predicho, prediccion[0]

print("✅ Función 'clasificar_imagen_personalizada()' definida")
print("\n📝 Uso: clasificar_imagen_personalizada('ruta/a/tu/imagen.png')")

## 11. Instrucciones para Usar tu Propia Imagen

### Opción 1: Dibujar en Paint o similar

1. Abre **Paint** (o cualquier programa de dibujo)
2. Dibuja un dígito grande y claro
3. Guarda la imagen como `mi_digito.png` en la misma carpeta que este notebook
4. Ejecuta la celda de abajo

### Opción 2: Usar Google Colab para dibujar interactivamente

Si usas Google Colab, puedes usar esta celda para dibujar directamente:

```python
from google.colab import files
from io import BytesIO

# Subir imagen desde tu computadora
uploaded = files.upload()
for filename in uploaded.keys():
    clasificar_imagen_personalizada(filename)
```

### Consejos para mejores resultados:

- ✅ Dibuja el dígito **grande** y **centrado**
- ✅ Usa trazo **grueso** y **continuo**
- ✅ Fondo **blanco** con dígito **negro** (o viceversa)
- ❌ Evita líneas muy finas o dígitos muy pequeños

### Ejemplo: Clasificar una Imagen del Dataset de Prueba

In [ ]:
# Primero, guardemos una imagen de ejemplo del dataset de prueba
idx_ejemplo = np.random.randint(0, len(X_test))
img_ejemplo = X_test[idx_ejemplo]

# Guardamos la imagen
Image.fromarray(img_ejemplo).save('ejemplo_digito_test.png')
print(f"✅ Guardada imagen de ejemplo: dígito {y_test[idx_ejemplo]}")

# Ahora la clasificamos usando nuestra función
digito, probs = clasificar_imagen_personalizada('ejemplo_digito_test.png')
print(f"\n🎯 El modelo predijo: {digito}")
print(f"📊 Etiqueta real: {y_test[idx_ejemplo]}")

### Clasificar TU Imagen Personalizada

**Ejecuta esta celda después de crear y guardar tu imagen:**

In [ ]:
# Cambia 'mi_digito.png' por el nombre de tu archivo
ruta_mi_imagen = 'dos.png'

# Verificamos si el archivo existe
if os.path.exists(ruta_mi_imagen):
    print(f"📂 Archivo encontrado: {ruta_mi_imagen}\n")
    digito, probabilidades = clasificar_imagen_personalizada(ruta_mi_imagen)
    print(f"\n🎯 El modelo predijo que escribiste el dígito: {digito}")
    print(f"📈 Probabilidades por dígito:")
    for i, prob in enumerate(probabilidades):
        print(f"   {i}: {prob*100:5.2f}%")
else:
    print(f"❌ No se encontró el archivo '{ruta_mi_imagen}'")
    print("\n💡 Instrucciones:")
    print("   1. Dibuja un dígito en Paint o similar")
    print("   2. Guárdalo como 'mi_digito.png' en la misma carpeta que este notebook")
    print("   3. Ejecuta esta celda de nuevo")

## 12. Cargar Modelo Guardado (Para Usar en Otra Sesión)

In [ ]:
# Si quieres cargar el modelo en otra sesión, usa:
# from tensorflow.keras.models import load_model
# model = load_model('modelo_mnist_digitos.keras')
# print("✅ Modelo cargado correctamente")

## 13. Experimentación y Mejoras

### Experimentos Sugeridos:

1. **Cambiar la arquitectura:**
   - Añade más capas ocultas
   - Prueba con diferentes números de neuronas (256, 512)
   - Experimenta con diferentes valores de Dropout (0.1, 0.3, 0.5)

2. **Usar Redes Convolucionales (CNN):**
   - Las CNN son mejores para imágenes que las redes densas
   - Pueden alcanzar >99% de precisión en MNIST

3. **Data Augmentation:**
   - Rotar, escalar, desplazar imágenes durante el entrenamiento
   - Hace el modelo más robusto

4. **Diferentes optimizadores:**
   - SGD, RMSprop, AdaGrad
   - Ajustar learning rate

### Ejemplo de Red Convolucional (Bonus):

In [ ]:
# BONUS: Red Neuronal Convolucional (CNN) para MNIST
# Las CNN son más efectivas para imágenes porque preservan la estructura espacial

def crear_modelo_cnn():
    """
    Crea un modelo CNN para MNIST.
    Requiere imágenes en formato (28, 28, 1) en lugar de (784,)
    """
    model_cnn = models.Sequential([
        # Primera capa convolucional
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),
        
        # Segunda capa convolucional
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Aplanamos para capas densas
        layers.Flatten(),
        
        # Capas densas
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(10, activation='softmax')
    ])
    
    model_cnn.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model_cnn

# Para entrenar el modelo CNN, necesitas reshape de los datos:
# X_train_cnn = X_train_norm.reshape(-1, 28, 28, 1)
# X_test_cnn = X_test_norm.reshape(-1, 28, 28, 1)
# model_cnn = crear_modelo_cnn()
# history_cnn = model_cnn.fit(X_train_cnn, y_train_cat, epochs=10, batch_size=128, validation_split=0.1)

print("✅ Función crear_modelo_cnn() definida")
print("   Descomenta el código para entrenar una CNN (alcanza ~99% de precisión)")

## 14. Resumen y Conclusiones

### ¿Qué hemos aprendido?

1. ✅ **Cargar y explorar** el dataset MNIST
2. ✅ **Preprocesar datos**: normalización, reshape, one-hot encoding
3. ✅ **Construir una red neuronal** con capas densas
4. ✅ **Entrenar el modelo** con validación
5. ✅ **Evaluar el rendimiento** con métricas y visualizaciones
6. ✅ **Guardar y cargar** modelos entrenados
7. ✅ **Clasificar imágenes personalizadas** dibujadas por ti

### Resultados Típicos:

- **Precisión en entrenamiento**: ~98-99%
- **Precisión en prueba**: ~97-98%
- **Tiempo de entrenamiento**: 1-3 minutos (CPU)

### Próximos Pasos:

- 🔹 Experimenta con diferentes arquitecturas
- 🔹 Prueba con otros datasets (Fashion-MNIST, CIFAR-10)
- 🔹 Implementa redes convolucionales (CNN)
- 🔹 Explora técnicas de transfer learning
- 🔹 Crea una aplicación web para clasificar dígitos en tiempo real

---

**¡Felicidades! Has creado tu primer clasificador de dígitos manuscritos con Deep Learning.** 🎉